# TEST FOR ANYLOGISTIX API
THe API info is here: https://anylogistix.help/api/python/readme.html

## PHASE 1 CONNECTION AND SIMULATION WITH THE ANYLOGISTIX API

## Install the dependencies
It assumes that the openapi_client is in the local directory ./openapi-python-client-3.3.1

In [111]:
import sys
import os
# This is my personal directory
os.chdir(r"C:\Users\Luis\Downloads\TFG\API")
#!{sys.executable} -m pip install "pydantic>=2.10,<3"
!{sys.executable} -m pip install "pydantic<2,>=1.10.5"
!{sys.executable} -m pip install -e ./openapi-python-client-3.3.1


Obtaining file:///C:/Users/Luis/Downloads/TFG/API/openapi-python-client-3.3.1
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for openapi-client (pyproject.toml): started
  Building editable for openapi-client (pyproject.toml): finished with status 'done'
  Created wheel for openapi-client: filename=openapi_client-1.0.0-0.editable-py3-none-any.whl size=3042 sha256=585c038b42c6c09146da41c8909914c9e7e29dac2ab14218b8116367a0f2c192
  Stored in directory: C:\Users\Luis\AppData\Local\Temp\pip-ephem-wheel-cache-gb4y47n9\wheels\96\67\

In [112]:
import openapi_client
from openapi_client.rest import ApiException
from pprint import pprint
import urllib3

In [113]:
# SERVER_IP="192.168.64.159"
# SERVER_IP="192.168.67.110"
SERVER_IP="alxserver.aut.uah.es"
SERVER_URL=f"https://{SERVER_IP}:443/api/v1"

# This is my personal APY KEY
API_KEY="c184f1ab-9f13-484c-a1c1-3d543502da6e"

SERVER_URL

'https://alxserver.aut.uah.es:443/api/v1'

## PART I: CHECK CONNECTIVIY AND OPERATIONAL STATUS

In [114]:
# Defining the host is optional and defaults to https://server_address:port/api/v1
# See configuration.py for a list of all supported configuration parameters.
configuration = openapi_client.Configuration(
    host = SERVER_URL
)
# The client must configure the authentication and authorization parameters
# in accordance with the API server security policy.
# Examples for each auth method are provided below, use the example that
# satisfies your auth use case.
# Configure API key authorization: ApiKey
configuration.api_key['ApiKey'] = API_KEY

# ALVARO: SUPER IMPORTANTE PARA PODER TRABAJAR CON CERTIFICADOS AUTO-FIRMADOS
configuration.verify_ssl = False
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
with openapi_client.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = openapi_client.OpenApiApi(api_client)

### Obtain current user data

In [115]:
try:
    # Gets current user data.
    api_response = api_instance.get_current_user()
    print("The response of OpenApiApi->get_current_user:\n")
    pprint(api_response)
except Exception as e:
    print("Exception when calling OpenApiApi->get_current_user: %s\n" % e)

The response of OpenApiApi->get_current_user:

ApiUserData(email='adrian.encabo@edu.uah.es', first_name='adrian.encabo@edu.uah.es', id=403, last_name='', username='adrian.encabo@edu.uah.es')


## PART II: OBTAIN PROJECTS, SCENARIOS, EXECUTIONS

### Obtain the list of currently available projects

In [116]:
try:
    # Returns a list of projects that the user has access to.
    api_response = api_instance.get_projects()
    print("The response of OpenApiApi->get_projects:\n")
    pprint(api_response)
except Exception as e:
    print("Exception when calling OpenApiApi->get_projects: %s\n" % e)

The response of OpenApiApi->get_projects:

[ApiProjectResponse(accessible=True, creation_date='2026-03-10T16:19:55.145394Z', current_user_id=403, deleted=False, id=79020, name='TFG_ADRIAN_ENCABO', owner_user_id=3)]


### Open a project and get the list of scenarios available

In [117]:
# My personal Project defined
MY_PROJECT_NAME="TFG_ADRIAN_ENCABO"
the_project_id=0
try:
    full_match = True # bool | Whether to match project name by complete match. (default to True)
    # Opens project with name projectName.
    the_project = api_instance.find_and_open_project_by_name(full_match, MY_PROJECT_NAME)
    print("The response of OpenApiApi->find_and_open_project_by_name:\n")
    pprint(the_project)
except Exception as e:
    print("Exception when calling OpenApiApi->find_and_open_project_by_name: %s\n" % e)
try:
    # Returns a list of scenarios for the project with identifier project_id.
    the_scenarios = api_instance.get_scenarios(the_project.id)
    print("The response of OpenApiApi->get_scenarios:\n")
    pprint(the_scenarios)
except Exception as e:
    print("Exception when calling OpenApiApi->get_scenarios: %s\n" % e)

The response of OpenApiApi->find_and_open_project_by_name:

ApiProjectResponse(accessible=True, creation_date='2026-03-10T16:19:55.145394Z', current_user_id=403, deleted=False, id=79020, name='TFG_ADRIAN_ENCABO', owner_user_id=3)
The response of OpenApiApi->get_scenarios:

[ApiScenarioData(id=82840, name='Budget Comparison (Estimated Demand)', type='SIM'),
 ApiScenarioData(id=87083, name='P3.3. GFA 1. Results 2_Different Locations', type='SIM'),
 ApiScenarioData(id=84191, name='Budget Comparison (Estimated Demand) 2', type='SIM'),
 ApiScenarioData(id=85538, name='Budget Comparison (Estimated Demand) 2', type='SIM'),
 ApiScenarioData(id=86671, name='P4.2.2 NO Distribution Network inside 4 Walls Models Capacity Restriction', type='NO'),
 ApiScenarioData(id=88185, name='P4.2.2 NO Distribution Network inside 4 Walls Models Capacity Restriction 2', type='NO'),
 ApiScenarioData(id=89212, name='P4.2.2 NO Distribution Network inside 4 Walls Models Capacity Restriction sim', type='SIM'),
 ApiSc

### Select the scenario by name

In [118]:
#Add the name of the scenario 
MY_SCENARIO_NAME="P4.2.6 SIM Distribution Network-4 Walls_New Inventory+New Transportation"

# 1. Select the scenario
the_scenario = next(s for s in the_scenarios if s.name == MY_SCENARIO_NAME)
pprint(the_scenario)

ApiScenarioData(id=92735, name='P4.2.6 SIM Distribution Network-4 Walls_New Inventory+New Transportation', type='SIM')


### Get the list of experiment runs of a scenario

In [119]:
try:
    # Returns the results of experiment runs for scenario.
    the_runs = api_instance.get_experiment_runs_for_scenario(the_scenario.id)
    print("The response of OpenApiApi->get_experiment_runs_for_scenario:\n")
    pprint(the_runs)
except Exception as e:
    print("Exception when calling OpenApiApi->get_experiment_runs_for_scenario: %s\n" % e)

The response of OpenApiApi->get_experiment_runs_for_scenario:

[ApiBasicRun(has_options=False, id=93824, name='Statistics', rc_id=93825, removing=False, scenario_id=92735, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=94035, name='Statistics 2', rc_id=94036, removing=False, scenario_id=92735, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=94246, name='Statistics 3', rc_id=94247, removing=False, scenario_id=92735, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=94433, name='Statistics 4', rc_id=94434, removing=False, scenario_id=92735, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=94629, name='Statistics 5', rc_id=94630, removing=False, scenario_id=92735, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=94806, name='Statistics 6', rc_id=94807, removing=False, scenario_id=92735, type='SIMULATION'),
 ApiBasicRun(has_options=False, id=181987, name='Statistics 7', rc_id=181988, removing=False, scenario_id=92735, type='SIMULATION'),
 ApiBasicRun(has_opt

## PART III. EXECUTE A SIMULATION

### Get the list of RunConfigurations for the experiments of a given Scenario
Each type of experiment has a RunConfiguration. For example:
```
ApiBasicRunConfiguration(animation_model_id=None, feasible_check_status='NOT_CONDUCTED', id=2439, is_feasible_check_data_available=False, model_time=None, name='i18n-Experiment_Simulation_text', progress=None, project_id=63, routes_progress=None, run_status=None, scenario_id=1752, speed=None, status='IDLE', step='IDLE', type='SIMULATION', validation_status='VALID'
```
Types of available experiment runs are:
* SIMULATION
* VARIATION
* COMPARISON
* SAFETY_STOCK
* RISK_ANALYSIS

In [120]:
MY_RUN_TYPE="SIMULATION"
try:
    # Returns a list of experiments available for a given scenario.
    the_run_configurations= api_instance.get_experiments(the_project.id, the_scenario.id)
    print("The response of OpenApiApi->get_experiments:\n")
    # Exception fixed for undefined variable
    pprint(the_run_configurations)
except Exception as e:
    print("Exception when calling OpenApiApi->get_experiments: %s\n" % e)

# 2. Select the adequate RunConfiguration for the scenario
the_RC = next((r for r in the_run_configurations if r.type == MY_RUN_TYPE), None)
pprint(the_RC)

The response of OpenApiApi->get_experiments:

[ApiBasicRunConfiguration(animation_model_id=None, feasible_check_status='NOT_CONDUCTED', id=93422, is_feasible_check_data_available=False, model_time=None, name='i18n-Experiment_Simulation_text', progress=None, project_id=79020, routes_progress=None, run_status=None, scenario_id=92735, speed=None, status='IDLE', step='IDLE', type='SIMULATION', validation_status='VALID'),
 ApiBasicRunConfiguration(animation_model_id=None, feasible_check_status='NOT_CONDUCTED', id=93579, is_feasible_check_data_available=False, model_time=None, name='i18n-Experiment_VariationExperiment_text', progress=None, project_id=79020, routes_progress=None, run_status=None, scenario_id=92735, speed=None, status='IDLE', step='IDLE', type='VARIATION', validation_status='NOT_CONDUCTED'),
 ApiBasicRunConfiguration(animation_model_id=None, feasible_check_status='NOT_CONDUCTED', id=92743, is_feasible_check_data_available=False, model_time=None, name='i18n-Experiment_Compariso

### Run a new experiment over a scenario (synch)
Runs synchronously.


In [121]:
try:
    # Starts the experiment synchronously.
    the_sync_result = api_instance.run_experiment_synchronously(the_RC.id)
    print("The response of OpenApiApi->run_experiment_synchronously:\n")
    pprint(the_sync_result)
except Exception as e:
    print("Exception when calling OpenApiApi->run_experiment_synchronously: %s\n" % e)

The response of OpenApiApi->run_experiment_synchronously:

ApiExperimentResult(experiment_result_id=257528, validation_errors=None, validation_status='OK', validation_warnings=None)


## Part IIIb. Asynch execution

### Run a new experiment over a scenario (asynch)
Runs asynchronously.
Periodic status check.

In [122]:
skip_scenario_warnings = True # bool | Whether to skip scenario warnings. (optional)
try:
    # Starts the experiment asynchronously.
    the_execution = api_instance.run_experiment(the_RC.id, skip_scenario_warnings=skip_scenario_warnings)
    print("The response of OpenApiApi->run_experiment:\n")
    pprint(the_execution)
except Exception as e:
    print("Exception when calling OpenApiApi->run_experiment: %s\n" % e)

import time
while True:
    try:
        # Returns the experiment status
        the_status = api_instance.get_experiment_status(the_RC.id)
        # pprint(api_response)

        # Check if execution finished
        if the_status.finished:
            print("Execution finished.")
            break

        # Optional: stop if failed
        # fixed exception of status check
        if the_status.failed:
            print("Execution failed.")
            break

    except Exception as e:
        print("Exception when calling OpenApiApi->get_experiment_status:", e)
    print("Not finished yet. Wait 1sec and check again")
    # wait before next check
    time.sleep(1)

The response of OpenApiApi->run_experiment:

ApiRunExperiment(animation_model_id=93422, experiment_type_id='SIMULATION', is3d_animation=False)
Execution finished.


### Get the experiments results (pages)
You can ask for limited number of results (avoid overflow)

In [123]:
skip = 0 # int | Number of records to skip. (default to 0)
take = 1000 # int | Number of records to be retrieved. (default to 1000)

try:
    # Returns all the results of the specific experiment run.
    the_run_results = api_instance.get_experiment_run_result(the_RC.id, skip, take)
    print("The response of OpenApiApi->get_experiment_run_result:\n")
    pprint(the_run_results)
except Exception as e:
    print("Exception when calling OpenApiApi->get_experiment_run_result: %s\n" % e)


The response of OpenApiApi->get_experiment_run_result:

ApiExperimentResultData(pages=[ApiExperimentResultPageWrapper(charts=[ApiExperimentResultGraphChartWrapper(chart=ApiChartMetadataShort(id=93425, is_group_mode_enabled=False, layout=ApiChartLayoutData(height=1, start_col=0, start_row=0, width=2), name='Transportation Cost, Revenue, Total Cost, Profit, Inventory Carrying Cost', rc_id=93422, type='BAR'), class_name='ApiExperimentResultGraphChartWrapper', data=ApiGraphChartData(entity_list=[], id=93425, show_x=True, total_entity_count=0, total_technical_row_count=0, type='BAR')), ApiExperimentResultGraphChartWrapper(chart=ApiChartMetadataShort(id=93426, is_group_mode_enabled=False, layout=ApiChartLayoutData(height=1, start_col=2, start_row=0, width=2), name='ELT Service Level by Products', rc_id=93422, type='LINE'), class_name='ApiExperimentResultGraphChartWrapper', data=ApiGraphChartData(entity_list=[], id=93426, show_x=True, total_entity_count=0, total_technical_row_count=0, type='L

In [124]:
for p in the_run_results.pages:
    print(f"\n{p.page.name} (id:{p.page.id})")
    for c in p.charts:
        print(f"  - {c.chart.name} (type: {c.chart.type}) (id: {c.chart.id})")


Financial and customer performance (id:93424)
  - Transportation Cost, Revenue, Total Cost, Profit, Inventory Carrying Cost (type: BAR) (id: 93425)
  - ELT Service Level by Products (type: LINE) (id: 93426)
  - Lead time (type: LINE) (id: 93427)
  - Lead time (type: HISTOGRAM) (id: 93428)
  - Transportation Cost, Revenue, Total Cost, Profit, Inventory Carrying Cost (type: TABLE) (id: 93429)
  - Demand (Orders Backlog), Fulfillment (Late Orders), Fulfillment Received (Orders On-time), Fulfillment Received (Orders) by Customer (type: TABLE) (id: 93430)

Operational performance  (id:93431)
  - Peak Capacity (type: HISTOGRAM) (id: 93432)
  - Peak Capacity (type: LINE) (id: 93433)
  - Available Inventory (type: LINE) (id: 93434)
  - Available Inventory in Product Units (type: LINE) (id: 93435)
  - Available Inventory in Product Units (type: LINE) (id: 93436)
  - Available Inventory in Product Units (type: HISTOGRAM) (id: 93437)

Inventory and capacity dynamics (id:93438)
  - Available Inve

### Obtain the pages in the experiment's result dashboard

In [125]:
try:
    # Returns statistics pages for the result of experiment run.
    # Exception fixed for 'id' attribute
    # We use the experiment_result_id from the previously executed synchronous run
    result_id = the_sync_result.experiment_result_id
    dashboard_pages = api_instance.get_experiment_dashboard_pages(result_id)
    print("The response of OpenApiApi->get_experiment_dashboard_pages:\n")
    
    # CHANGED Displaying the available pages and charts with their dynamic IDs
    for page in dashboard_pages:
        print(f"Dashboard Page: {page.name} (ID: {page.id})")
        for chart in page.charts:
            print(f"  - Chart: {chart.name} (ID: {chart.id})")
            
except Exception as e:
    print("Exception when calling OpenApiApi->get_experiment_dashboard_pages: %s\n" % e)

The response of OpenApiApi->get_experiment_dashboard_pages:

Dashboard Page: Financial and customer performance (ID: 257530)
  - Chart: Transportation Cost, Revenue, Total Cost, Profit, Inventory Carrying Cost (ID: 257531)
  - Chart: ELT Service Level by Products (ID: 257532)
  - Chart: Lead time (ID: 257533)
  - Chart: Lead time (ID: 257534)
  - Chart: Transportation Cost, Revenue, Total Cost, Profit, Inventory Carrying Cost (ID: 257535)
  - Chart: Demand (Orders Backlog), Fulfillment (Late Orders), Fulfillment Received (Orders On-time), Fulfillment Received (Orders) by Customer (ID: 257536)
Dashboard Page: Operational performance  (ID: 257537)
  - Chart: Peak Capacity (ID: 257538)
  - Chart: Peak Capacity (ID: 257539)
  - Chart: Available Inventory (ID: 257540)
  - Chart: Available Inventory in Product Units (ID: 257541)
  - Chart: Available Inventory in Product Units (ID: 257542)
  - Chart: Available Inventory in Product Units (ID: 257543)
Dashboard Page: Inventory and capacity dyna

### Export the results as an excel matrix

In [126]:
# Exception fixed for wrong IDs
# EXPORT ALL DASHBOARD PAGES TO EXCEL
import shutil #For manipulating excel files
import os
import re # For cleaning strings

try:
    # Dynamically select the first dashboard page to export to avoid hardcoded ID errors
    if dashboard_pages and len(dashboard_pages) > 0:
        target_result_id = the_sync_result.experiment_result_id
        print(f"Exporting ALL dashboard pages from result ID {target_result_id}...\n")
        
        # Iterate over every page available in the dashboard
        for page in dashboard_pages:
            
            # Clean the page name to make it a safe Windows filename (replace spaces with underscores)
            safe_page_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', page.name)

            # Save the exported Excel file locally
            # Create a unique filename for each page
            output_filename = f"Simulation_Results_{safe_page_name}.xlsx"
            output_path = os.path.join(os.getcwd(), output_filename)
            
            print(f"Downloading '{page.name}' (ID: {page.id}) -> {output_filename}")

            # Returns an Excel representation of the dashboard page
            # Export this specific page
            excel_export = api_instance.export_dashboard_page(target_result_id, page.id)

            # Check if the API returns a temporary file path
            # Save the file securely
            if isinstance(excel_export, str) and os.path.exists(excel_export):
                shutil.move(excel_export, output_path)
            elif isinstance(excel_export, bytes):
                with open(output_path, "wb") as file:
                    file.write(excel_export)
            else:
                print(f"Unexpected data format received for page {page.name}")
                
        print("\n[SUCCESS] All dashboard pages exported successfully.")
    else:
        print("No dashboard pages available to export.")
        
except Exception as e:
    print("Exception when calling OpenApiApi->export_dashboard_page: %s\n" % e)

Exporting ALL dashboard pages from result ID 257528...


[SUCCESS] All dashboard pages exported successfully.


### Close current project

In [127]:
#Error fixed of indentation
try:
    # Closes the currently open project.
    api_response = api_instance.close_project()
    print("The response of OpenApiApi->close_project:\n")
    pprint(api_response)
    print("\nProject closed successfully.")
except ApiException as e:
    print("Exception when calling OpenApiApi->close_project: %s\n" % e)

The response of OpenApiApi->close_project:

ApiProjectResponse(accessible=True, creation_date='2026-03-10T16:19:55.145394Z', current_user_id=None, deleted=False, id=79020, name='TFG_ADRIAN_ENCABO', owner_user_id=3)

Project closed successfully.


## ==========================================================
# PHASE 2

## PART I Flat AI Decision & Scenario Cloning

## Install the dependencies

In [134]:
!pip install pandas openpyxl

### 1. Define the Flat AI logic (3 predefined decisions)

In [129]:
decision_options = [
    "DECISION 0: Increase Demand by 20% (Aggressive Growth)",
    "DECISION 1: Decrease Transport Costs by 15% (Cost Optimization)",
    "DECISION 2: Increase Safety Stock by 10% (Conservative Risk Management)"
]

print("--- FLAT AI: EVALUATING PREDEFINED DECISIONS ---")
for i, decision in enumerate(decision_options):
    print(f"[{i}] {decision}")

# For this 'flat AI' phase, we will simulate the AI choosing a decision automatically.
# Let us assume the AI evaluates the current context and selects Decision 0.
selected_decision_index = 1

selected_decision_text = decision_options[selected_decision_index]

print(f"\nFLAT AI SELECTED: {selected_decision_text}")

--- FLAT AI: EVALUATING PREDEFINED DECISIONS ---
[0] DECISION 0: Increase Demand by 20% (Aggressive Growth)
[1] DECISION 1: Decrease Transport Costs by 15% (Cost Optimization)
[2] DECISION 2: Increase Safety Stock by 10% (Conservative Risk Management)

FLAT AI SELECTED: DECISION 1: Decrease Transport Costs by 15% (Cost Optimization)


### 2. Flat AI Autonomous Data Modification (FAILED)

In [102]:
# The AI reads the clean base Excel template, dynamically locates the 'Quantity' 
# parameter across AnyLogistix 'Col X' format, applies +20%, forces INTEGER values,
# and saves a new file.
# Pandas acts as a radar to find coordinates, OpenPyXL injects the data.
# (ADAPTED: Fully reverting to pure Pandas with strict Integer formatting to satisfy ALX validation)

import pandas as pd
import os

# 1. Define the input and output paths
base_excel_path = r"C:\Users\Luis\Downloads\TFG\API\P4.2.6 SIM Distribution Network-4 Walls_New Inventory+New Transportation.xlsx" 
modified_excel_path = os.path.join(os.getcwd(), "AI_Modified_Scenario.xlsx")

# AI Logic Mapping based on selected decision
if selected_decision_index == 0:
    target_sheet = 'Demand'
    target_param_name = 'Quantity'
    value_offset = 3      
    multiplier = 1.20     
    enforce_int = True    
elif selected_decision_index == 1:
    target_sheet = 'Paths'
    target_param_name = 'Cost per unit'
    value_offset = 1      
    multiplier = 0.85     
    enforce_int = False   
elif selected_decision_index == 2:
    target_sheet = 'Inventory'
    target_param_name = 'Safety stock'
    value_offset = 1      
    multiplier = 1.10     
    enforce_int = True    

print(f"--- PHASE 2 - BLOCK 2: AI APPLYING DECISION {selected_decision_index} ---")

try:
    if os.path.exists(base_excel_path):
        print("1. Scanning and modifying database with Pandas...")
        # Load all sheets into memory
        all_sheets = pd.read_excel(base_excel_path, sheet_name=None)
        df = all_sheets.get(target_sheet)
        
        if df is not None:
            # Counter to track how many rows the AI modified
            modifications_made = 0

            # AnyLogistix stores parameter names in Col X, and numerical values in Col X+3
            for i in range(1, 20):
                name_col = f'Col {i}'
                val_col = f'Col {i + value_offset}'
                
                if name_col in df.columns and val_col in df.columns:
                    mask = df[name_col].astype(str).str.strip() == target_param_name
                    
                    if mask.any():
                        # We found quantities! Apply the 20% increase mathematically
                        # We force numeric conversion first to ensure safe multiplication
                        numeric_values = pd.to_numeric(df.loc[mask, val_col], errors='coerce')
                        new_values = numeric_values * multiplier
                        
                        # STRICT FORMATTING TO PREVENT ALX REJECTION
                        if enforce_int:
                            new_values = new_values.fillna(0).round().astype(int).astype(object)
                        else:
                            new_values = new_values.astype(object)
                            
                        # CRITICAL FIX: AnyLogistix expects STRICT INTEGERS for physical demand.
                        # We must round the floats (14.4 -> 14) and cast to int to prevent silent rejection.    
                        df.loc[mask, val_col] = new_values
                        # Add the number of modified rows to our tracker
                        modifications_made += mask.sum()
            
            if modifications_made > 0:
                # Replace the original DataFrame with our modified one
                all_sheets[target_sheet] = df
                print(f"-> AI successfully modified {modifications_made} rows in memory.")
                
                print("2. Saving the modified scenario Excel...")
                # Save all sheets back to a new Excel file
                with pd.ExcelWriter(modified_excel_path, engine='openpyxl') as writer:
                    for sheet_name, sheet_df in all_sheets.items():
                        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"[SYSTEM SUCCESS] Decision {selected_decision_index} securely injected into: {modified_excel_path}")
            else:
                print(f"Error: Could not find '{target_param_name}' in '{target_sheet}'.")
        else:
            print(f"Error: Sheet '{target_sheet}' not found.")
    else:
        print(f"Error: Base Excel file not found at {base_excel_path}")

except PermissionError:
    print("\n[ERROR DE PERMISOS] El archivo 'AI_Modified_Scenario.xlsx' está abierto.")
    print("Por favor, cierra Microsoft Excel y vuelve a ejecutar esta celda.")
except Exception as e:
    print(f"Exception during AI Universal Excel modification: {e}")

--- PHASE 2 - BLOCK 2: AI APPLYING DECISION 0 ---
1. Scanning and modifying database with Pandas...


C:\Users\Luis\AppData\Local\Temp\ipykernel_17404\1768921875.py:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[14 44 17 11 1 19 16 140 17 28 1 61 7 41 11 91 7 26 28 12 43 12 10 12 52
 32 12 12 13 268 25 156 28 12 4 41 11 11 10 20 30 16 131 7 37 8 7 10 2 132
 20 41 42 24 17 11 8 32 10 124 14 132 17 29 13 6 13 26 31 10 22 10 32 34
 92 2 90 47 8 11 38 17 8 50 14 7 14 17 13 124 116 89 96 55 29 18 23 19 151
 26 11 97 4 8 12 12 10 25 22 18 18 8 11 16 26 17 72 1 28 12 60 8 89 8 85
 31 19 35 13 32 13 16 10 12 16 58 11 120 7 124 19 34 55 16 1 31 10 10 19
 20 2 16 8]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[mask, val_col] = new_values


-> AI successfully modified 153 rows in memory.
2. Saving the modified scenario Excel...
[SYSTEM SUCCESS] Decision 0 securely injected into: C:\Users\Luis\Downloads\TFG\API\AI_Modified_Scenario.xlsx


### 2. Flat AI Autonomous Data Modification (SURGICAL METHOD) (FAILED)

In [104]:
# The AI uses openpyxl to surgically edit ONLY the specific values, retaining 
# 100% of AnyLogistix's hidden metadata, formatting, and internal database structures.
# 100% Pure OpenPyXL approach. No Pandas means no coordinate mismatching.

import openpyxl
import os

# Define the input and output paths
base_excel_path = r"C:\Users\Luis\Downloads\TFG\API\Global Network Examination.xlsx" 
modified_excel_path = os.path.join(os.getcwd(), "AI_Modified_Scenario.xlsx")

# AI Logic Mapping based on selected decision
if selected_decision_index == 0:
    target_sheet = 'Demand'
    target_param_name = 'Quantity'
    value_offset = 3      
    multiplier = 1.20     
    enforce_int = True    
elif selected_decision_index == 1:
    target_sheet = 'Paths'
    target_param_name = 'Cost per unit'
    value_offset = 1      
    multiplier = 0.85     
    enforce_int = False   
elif selected_decision_index == 2:
    target_sheet = 'Inventory'
    target_param_name = 'Safety stock'
    value_offset = 1      
    multiplier = 1.10     
    enforce_int = True    

print(f"--- PHASE 2 - BLOCK 2: AI APPLYING DECISION {selected_decision_index} ---")

try:
    # Load workbook retaining all internal ALX formats
    wb = openpyxl.load_workbook(base_excel_path)
    if target_sheet in wb.sheetnames:
        ws = wb[target_sheet]
        
        # 1. Exact native Header Locator
        # Map column names to their exact numerical indexes
        header = {}
        header_row_idx = 1  # Row 1 is the header
        # Iterate through all data rows
        for r in range(1, 10): 
            row_vals = [str(ws.cell(row=r, column=c).value).strip() for c in range(1, ws.max_column + 1) if ws.cell(row=r, column=c).value is not None]
            if 'ID' in row_vals and any(col.startswith('Col ') for col in row_vals):
                header_row_idx = r
                for c in range(1, ws.max_column + 1):
                    val = ws.cell(row=r, column=c).value
                    if val is not None:
                        header[str(val).strip()] = c
                break
                
        print(f"1. Scanning coordinates natively... (Header found at row {header_row_idx})")
        
        # 2. Perfect Surgical Modification
        modifications_made = 0
        for r in range(header_row_idx + 1, ws.max_row + 1):
            # Check ALX's Col X format
            for i in range(1, 20):
                name_col = f'Col {i}'
                val_col = f'Col {i + value_offset}'
                
                if name_col in header and val_col in header:
                    name_c = header[name_col]
                    val_c = header[val_col]
                    
                    cell_name_val = str(ws.cell(row=r, column=name_c).value).strip()
                    if cell_name_val == target_param_name:
                        old_val = ws.cell(row=r, column=val_c).value
                        if old_val is not None:
                            try:
                                numeric_val = float(old_val)
                                new_val = numeric_val * multiplier
                                if enforce_int:
                                    new_val = int(round(new_val))
                                ws.cell(row=r, column=val_c).value = new_val
                                modifications_made += 1
                            except ValueError:
                                pass
        
        if modifications_made > 0:
            # Save the surgical modification
            wb.save(modified_excel_path)
            print(f"[SYSTEM SUCCESS] {modifications_made} modifications securely injected into: {modified_excel_path}")
        else:
            print(f"Error: Parameter '{target_param_name}' not found or modified.")
    else:
        print(f"Error: Sheet '{target_sheet}' not found.")
        
except PermissionError:
    print("\n[ERROR DE PERMISOS] El archivo 'AI_Modified_Scenario.xlsx' está abierto en Excel. Ciérralo.")
except Exception as e:
    print(f"Exception during execution: {e}")

--- PHASE 2 - BLOCK 2: AI APPLYING DECISION 0 ---


C:\Users\Luis\Downloads\TFG\Anaconda\envs\tfg_anylogistix\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


1. Scanning coordinates natively... (Header found at row 1)
[SYSTEM SUCCESS] 153 modifications securely injected into: C:\Users\Luis\Downloads\TFG\API\AI_Modified_Scenario.xlsx


### 2. Flat AI Autonomous Data Modification (SURGICAL METHOD) (FAILED)

In [ ]:
# PHASE 2 - BLOCK 2: AI AUTONOMOUS DATA MODIFICATION (PURE NATIVE)
# The AI uses openpyxl to safely modify ONLY the target cells, leaving 
# 100% of the ALX structural metadata, hidden sheets, and formats intact.

import openpyxl
import os

base_excel_path = r"C:\Users\Luis\Downloads\TFG\API\Global Network Examination.xlsx" 
modified_excel_path = os.path.join(os.getcwd(), "AI_Modified_Scenario.xlsx")

# AI Logic Mapping based on selected decision
if selected_decision_index == 0:
    target_sheet = 'Demand'
    target_param_name = 'Quantity'
    value_offset = 3      
    multiplier = 1.20     
    enforce_int = True    
elif selected_decision_index == 1:
    target_sheet = 'Paths'
    target_param_name = 'Cost per unit'
    value_offset = 1      
    multiplier = 0.85     # Reduce transport costs by 15%
    enforce_int = False   
elif selected_decision_index == 2:
    target_sheet = 'Inventory'
    target_param_name = 'Safety stock'
    value_offset = 1      
    multiplier = 1.10     
    enforce_int = True    

print(f"--- PHASE 2 - BLOCK 2: AI APPLYING DECISION {selected_decision_index} ---")

try:
    if os.path.exists(base_excel_path):
        print(f"1. Loading base scenario natively from: {base_excel_path}")
        # data_only=False ensures formulas and structural formats are kept intact
        wb = openpyxl.load_workbook(base_excel_path)
        
        if target_sheet in wb.sheetnames:
            print(f"2. Found '{target_sheet}' table. Modifying parameters safely...")
            ws = wb[target_sheet]
            modifications_made = 0
            
            # Iterate through all rows
            for row in range(2, ws.max_row + 1):
                # Search horizontally across the row to find the exact parameter name
                for col in range(1, ws.max_column + 1):
                    cell_val = ws.cell(row=row, column=col).value
                    
                    if str(cell_val).strip() == target_param_name:
                        # Found the parameter! The number is located at 'value_offset' columns to the right
                        target_col = col + value_offset
                        value_cell = ws.cell(row=row, column=target_col)
                        
                        old_val = value_cell.value
                        
                        # Verify the cell contains a number before doing math
                        if old_val is not None and isinstance(old_val, (int, float)):
                            new_val = float(old_val) * multiplier
                            
                            if enforce_int:
                                new_val = int(round(new_val))
                                
                            value_cell.value = new_val
                            modifications_made += 1
                            break # Done with this row, move to the next
            
            if modifications_made > 0:
                print(f"Success! Parameter '{target_param_name}' mathematically modified in {modifications_made} rows.")
                
                # Update internal Scenario Name to avoid server conflicts
                if 'Scenario settings' in wb.sheetnames:
                    ws_settings = wb['Scenario settings']
                    for r in range(1, ws_settings.max_row + 1):
                        if str(ws_settings.cell(row=r, column=1).value).strip() == 'Name':
                            ws_settings.cell(row=r, column=2).value = f"AI_Scenario_Dec_{selected_decision_index}"
                            break

                print(f"3. Saving the modified scenario to: {modified_excel_path}")
                wb.save(modified_excel_path)
                print("\n[SYSTEM] AI safely modified the file without breaking its structure!")
            else:
                print(f"Error: Could not find '{target_param_name}' or no numeric values to modify.")
        else:
            print(f"Error: Sheet '{target_sheet}' not found in Excel.")
    else:
        print(f"Error: Base Excel file not found at {base_excel_path}")

except PermissionError:
    print("\n[ERROR DE PERMISOS] El archivo 'AI_Modified_Scenario.xlsx' está abierto. Ciérralo en Excel y reintenta.")
except Exception as e:
    print(f"Exception during AI Native Excel modification: {e}")

### 2. Flat AI Autonomous Data Modification (UNIVERSAL METHOD)  (FAILED)

In [98]:
# Hybrid Approach: Pandas acts as a radar to find exact cell coordinates, 
# and OpenPyXL acts as a surgeon to inject the values safely into the ALX database.

import pandas as pd
import openpyxl
import os

# 1. Base Paths
base_excel_path = r"C:\Users\Luis\Downloads\TFG\API\Global Network Examination.xlsx" 
modified_excel_path = os.path.join(os.getcwd(), "AI_Modified_Scenario.xlsx")

# 2. Logic Mapping: The AI configures itself based on the selected decision
if selected_decision_index == 0:
    target_sheet = 'Demand'
    target_param_name = 'Quantity'
    value_offset = 3      # In Demand, value is in Col X+3
    multiplier = 1.20     # +20% Demand
    enforce_int = True    # Demand physical units must be integers
elif selected_decision_index == 1:
    target_sheet = 'Paths'
    target_param_name = 'Cost per unit'
    value_offset = 1      # In Paths, value is in Col X+1
    multiplier = 0.85     # -15% Transport Costs
    enforce_int = False   # Costs can be float decimals
elif selected_decision_index == 2:
    target_sheet = 'Inventory'
    target_param_name = 'Safety stock'
    value_offset = 1      # In Inventory, value is in Col X+1
    multiplier = 1.10     # +10% Safety Stock
    enforce_int = True    # Stock physical units must be integers 

print(f"--- PHASE 2 - BLOCK 2: AI APPLYING DECISION {selected_decision_index} ---")

try:
    if os.path.exists(base_excel_path):
        # 3. PHASE A: PANDAS AS A RADAR
        print("1. Scanning database with Pandas to find exact cell coordinates...")
        df = pd.read_excel(base_excel_path, sheet_name=target_sheet)
        cells_to_modify = [] 
        
        for i in range(1, 20):
            name_col = f'Col {i}'
            val_col = f'Col {i + value_offset}'
            
            if name_col in df.columns and val_col in df.columns:
                # Find rows matching the target parameter (cleaning empty spaces)
                mask = df[name_col].astype(str).str.strip() == target_param_name
                for idx in df[mask].index:
                    old_val = df.at[idx, val_col]
                    try:
                        numeric_val = float(old_val)
                        new_val = numeric_val * multiplier
                        if enforce_int:
                            new_val = int(round(new_val))

                        # Map Pandas coordinates to Excel coordinates
                        excel_row = idx + 2  # Pandas index 0 is Excel row 2
                        excel_col = df.columns.get_loc(val_col) + 1 
                        cells_to_modify.append((excel_row, excel_col, new_val))
                    except ValueError:
                        pass  # Skip non-numeric values
                        
        print(f"-> Radar detected {len(cells_to_modify)} target cells to modify.")

        # 4. PHASE B: OPENPYXL AS A SURGEON
        if len(cells_to_modify) > 0:
            print("2. Opening Excel surgically with OpenPyXL...")
            wb = openpyxl.load_workbook(base_excel_path)
            ws = wb[target_sheet]
            
            for r, c, val in cells_to_modify:
                ws.cell(row=r, column=c).value = val
                        
            wb.save(modified_excel_path)
            print(f"[SYSTEM SUCCESS] Decision securely injected into: {modified_excel_path}")
        else:
            print(f"Error: Radar could not find '{target_param_name}' in '{target_sheet}'.")
    else:
        print(f"Error: Base Excel file not found at {base_excel_path}")

except PermissionError:
    print("\n[ERROR DE PERMISOS] El archivo 'AI_Modified_Scenario.xlsx' está abierto.")
    print("Por favor, cierra Microsoft Excel y vuelve a ejecutar esta celda.")
except Exception as e:
    print(f"Exception during AI Universal Excel modification: {e}")

--- PHASE 2 - BLOCK 2: AI APPLYING DECISION 0 ---
1. Scanning database with Pandas to find exact cell coordinates...
-> Radar detected 153 target cells to modify.
2. Opening Excel surgically with OpenPyXL...


C:\Users\Luis\Downloads\TFG\Anaconda\envs\tfg_anylogistix\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


[SYSTEM SUCCESS] Decision securely injected into: C:\Users\Luis\Downloads\TFG\API\AI_Modified_Scenario.xlsx


### 2. Flat AI Autonomous Data Modification (FAILED)

In [130]:
# The AI reads the clean base Excel template, dynamically locates the 'Quantity' 
# or target parameter across AnyLogistix 'Col X' format, applies the logic, and saves a new file.

import pandas as pd
import os

base_excel_path = r"C:\Users\Luis\Downloads\TFG\API\Global Network Examination.xlsx" 
modified_excel_path = os.path.join(os.getcwd(), "AI_Modified_Scenario.xlsx")

# AI Logic Mapping based on selected decision
if selected_decision_index == 0:
    target_sheet = 'Demand'
    target_param_name = 'Quantity'
    value_offset = 3      
    multiplier = 1.20     
    enforce_int = True    
elif selected_decision_index == 1:
    target_sheet = 'Paths'
    target_param_name = 'Cost per unit'
    value_offset = 1      
    multiplier = 0.85     # Reduce transport costs by 15%
    enforce_int = False   
elif selected_decision_index == 2:
    target_sheet = 'Inventory'
    target_param_name = 'Safety stock'
    value_offset = 1      
    multiplier = 1.10     
    enforce_int = True    

print(f"--- PHASE 2 - BLOCK 2: AI APPLYING DECISION {selected_decision_index} ---")

try:
    if os.path.exists(base_excel_path):
        print(f"1. Loading base scenario from: {base_excel_path}")
        all_sheets = pd.read_excel(base_excel_path, sheet_name=None)
        df = all_sheets.get(target_sheet)
        
        if df is not None:
            print(f"2. Found '{target_sheet}' table. Searching for '{target_param_name}' parameters...")
            modifications_made = 0
            
            for i in range(1, 20):
                name_col = f'Col {i}'
                val_col = f'Col {i + value_offset}'
                
                if name_col in df.columns and val_col in df.columns:
                    mask = df[name_col].astype(str).str.strip() == target_param_name
                    if mask.any():
                        numeric_values = pd.to_numeric(df.loc[mask, val_col], errors='coerce')
                        new_values = numeric_values * multiplier
                        
                        if enforce_int:
                            new_values = new_values.fillna(0).round().astype(int).astype(object)
                        else:
                            new_values = new_values.astype(object)
                            
                        df.loc[mask, val_col] = new_values
                        modifications_made += mask.sum()
            
            if modifications_made > 0:
                all_sheets[target_sheet] = df
                print(f"Success! Parameter mathematically modified in {modifications_made} rows.")
                
                print(f"3. Saving the modified scenario to: {modified_excel_path}")
                with pd.ExcelWriter(modified_excel_path, engine='openpyxl') as writer:
                    for sheet_name, sheet_df in all_sheets.items():
                        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
                print("\n[SYSTEM] Flat AI completely automated the data modification!")
            else:
                print(f"Error: Could not find '{target_param_name}' in '{target_sheet}'.")
        else:
            print(f"Error: Sheet '{target_sheet}' not found.")
    else:
        print(f"Error: Base Excel file not found at {base_excel_path}")

except PermissionError:
    print("\n[ERROR DE PERMISOS] El archivo 'AI_Modified_Scenario.xlsx' está abierto. Ciérralo.")
except Exception as e:
    print(f"Exception during AI Universal Excel modification: {e}")

--- PHASE 2 - BLOCK 2: AI APPLYING DECISION 1 ---
1. Loading base scenario from: C:\Users\Luis\Downloads\TFG\API\Global Network Examination.xlsx


C:\Users\Luis\Downloads\TFG\Anaconda\envs\tfg_anylogistix\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


2. Found 'Paths' table. Searching for 'Cost per unit' parameters...
Success! Parameter mathematically modified in 7 rows.
3. Saving the modified scenario to: C:\Users\Luis\Downloads\TFG\API\AI_Modified_Scenario.xlsx


C:\Users\Luis\AppData\Local\Temp\ipykernel_17404\3359651656.py:57: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.017 0.017 0.0425 0.0425 0.017 0.017 0.0425]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask, val_col] = new_values



[SYSTEM] Flat AI completely automated the data modification!


### 3. Modification by Excel (FAILED)

In [84]:
# ROBUST SYNC INJECTION (CLONE + OVERWRITE)
# To bypass ALX server bugs when creating scenarios from scratch, the AI synchronously 
# clones the base scenario, and then synchronously overwrites it with the modified Excel.

import os
from pprint import pprint
import openapi_client

print("--- PHASE 2 - BLOCK 3: CLONE AND OVERWRITE WITH AI DATA ---")

try:
    if os.path.exists(modified_excel_path):
        print(f"1. Cloning base scenario '{the_scenario.name}' synchronously...")
        new_clone_name = f"{the_scenario.name} - AI Decision 0"
        
        # Define the copy parameters with our custom name
        copy_parameters = openapi_client.ApiConvertAndCopyParameters(new_name=new_clone_name)
        
        cloned_scenario = api_instance.copy_scenario_synchronously(
            new_type=the_scenario.type, 
            scenario_id=the_scenario.id,
            body=copy_parameters
        )
        
        if cloned_scenario and cloned_scenario.id:
            print(f"   [SUCCESS] Scenario cloned! New ID: {cloned_scenario.id}")
            
            print(f"\n2. Injecting the modified Excel data into the clone synchronously...")
            print("   (This will safely overwrite the demand data with our +20% logic)")
            
            # We use the existing import function which is bulletproof
            updated_scenario = api_instance.import_scenario_existing(
                project_id=the_project.id,
                scenario_id=cloned_scenario.id,
                file=modified_excel_path
            )
            
            print(f"\n[SYSTEM] TOTAL SUCCESS! AI data injected perfectly.")
            print(f"Final AI Scenario ID: {updated_scenario.id}")
            print(f"Final AI Scenario Name: {updated_scenario.name}")
            
            # Save it globally for the final simulation block
            the_ai_scenario = updated_scenario
            
        else:
            print("Error: Failed to clone the scenario. Server returned empty data.")
    else:
        print(f"Error: Could not find the file {modified_excel_path}")
        
except Exception as e:
    print("\n[SERVER ERROR] Exception during clone or upload:")
    print(e)

--- PHASE 2 - BLOCK 3: CLONE AND OVERWRITE WITH AI DATA ---
1. Cloning base scenario 'Global Network Examination' synchronously...
   [SUCCESS] Scenario cloned! New ID: 243908

2. Injecting the modified Excel data into the clone synchronously...
   (This will safely overwrite the demand data with our +20% logic)

[SYSTEM] TOTAL SUCCESS! AI data injected perfectly.
Final AI Scenario ID: 243908
Final AI Scenario Name: AI_Modified_Scenario


### 3. Modification by Excel (FAILED)

In [99]:
# ROBUST SYNC INJECTION (CLONE + OVERWRITE)
# The AI synchronously clones the base scenario, and then synchronously 
# overwrites it with the healthy OpenPyXL modified data.

import os
from pprint import pprint
import openapi_client

print(f"--- PHASE 2 - BLOCK 3: CLONE AND OVERWRITE (DECISION {selected_decision_index}) ---")

try:
    if os.path.exists(modified_excel_path):
        print(f"1. Cloning base scenario '{the_scenario.name}' synchronously...")
        new_clone_name = f"{the_scenario.name} - AI Dec {selected_decision_index}"
        
        # Define the copy parameters with our custom name
        copy_parameters = openapi_client.ApiConvertAndCopyParameters(new_name=new_clone_name)
        
        cloned_scenario = api_instance.copy_scenario_synchronously(
            new_type=the_scenario.type, 
            scenario_id=the_scenario.id,
            body=copy_parameters
        )
        
        if cloned_scenario and cloned_scenario.id:
            print(f"   [SUCCESS] Scenario cloned! New ID: {cloned_scenario.id}")
            
            print(f"\n2. Injecting the modified Excel data into the clone synchronously...")
            # We use the existing import function which Updates the rows based on the healthy Excel
            updated_scenario = api_instance.import_scenario_existing(
                project_id=the_project.id,
                scenario_id=cloned_scenario.id,
                file=modified_excel_path
            )
            
            print(f"\n[SYSTEM] TOTAL SUCCESS! AI data injected perfectly.")
            print(f"Final AI Scenario ID: {updated_scenario.id}")
            
            # Save it globally for the final simulation block
            the_ai_scenario = updated_scenario
            
        else:
            print("Error: Failed to clone the scenario. Server returned empty data.")
    else:
        print(f"Error: Could not find the file {modified_excel_path}")
        
except Exception as e:
    print("\n[SERVER ERROR] Exception during clone or upload:")
    print(e)

--- PHASE 2 - BLOCK 3: CLONE AND OVERWRITE (DECISION 0) ---
1. Cloning base scenario 'Global Network Examination' synchronously...
   [SUCCESS] Scenario cloned! New ID: 249011

2. Injecting the modified Excel data into the clone synchronously...

[SYSTEM] TOTAL SUCCESS! AI data injected perfectly.
Final AI Scenario ID: 249011


### 3. Modification by Excel (Async injection) (FAILED)

In [89]:
# ASYNC INJECTION (THE DEFINITIVE METHOD)
# The AI uploads the modified Excel file to create a brand new database from scratch. 
# It bypasses the ALX naming bug by locating the newly created scenario by its maximum ID.

import os
import time

print("--- PHASE 2 - BLOCK 3: UPLOADING AI MODIFIED SCENARIO (ASYNC) ---")

try:
    if os.path.exists(modified_excel_path):
        print(f"1. Uploading file to AnyLogistix server: {modified_excel_path}")
        print("Starting asynchronous import...")
        
        # Start the import process in the background
        import_response = api_instance.import_excel(
            new_scenario_name="AI_Scenario", # The server might ignore this and name it 'null X'
            project_id=the_project.id,
            file=modified_excel_path
        )
        
        job_id = getattr(import_response, 'job_id', None)
        
        if job_id:
            print(f"Upload job started successfully. Job ID: {job_id}")
            print("2. Waiting for the server to build the database...")
            
            finished = False
            
            # Poll the server until it finishes processing the Excel
            while not finished:
                status_response = api_instance.get_import_status(job_id)
                current_status = getattr(status_response, 'status', 'UNKNOWN')
                
                if current_status == 'DONE':
                    finished = True
                elif current_status in ['FAILED', 'ERROR']:
                    print("The server failed to process the Excel file.")
                    finished = True
                else:
                    time.sleep(3) # Wait before checking again
                    
            if current_status == 'DONE':
                print("\n3. Import finished! Locating the new scenario ID in the database...")
                
                # Fetch all scenarios currently in the project
                scenarios = api_instance.get_scenarios(the_project.id)
                
                # THE FIX: Sort scenarios by ID descending and pick the very first one (the highest ID)
                # This perfectly bypasses the 'null X' naming bug of the ALX server.
                newest_scenario = sorted(scenarios, key=lambda x: x.id, reverse=True)[0]
                
                the_ai_scenario = newest_scenario
                
                print(f"\n[SYSTEM] TOTAL SUCCESS! AI Scenario created from scratch.")
                print(f"Final AI Scenario ID: {the_ai_scenario.id}")
                print(f"Assigned Server Name: {the_ai_scenario.name}")
                
        else:
            print("\nError: Could not extract job_id from the server response.")
            
    else:
        print(f"Error: Could not find the file {modified_excel_path}")
        
except Exception as e:
    print("Exception during Excel upload: %s\n" % e)

--- PHASE 2 - BLOCK 3: UPLOADING AI MODIFIED SCENARIO (ASYNC) ---
1. Uploading file to AnyLogistix server: C:\Users\Luis\Downloads\TFG\API\AI_Modified_Scenario.xlsx
Starting asynchronous import...
Upload job started successfully. Job ID: 94
2. Waiting for the server to build the database...

3. Import finished! Locating the new scenario ID in the database...

[SYSTEM] TOTAL SUCCESS! AI Scenario created from scratch.
Final AI Scenario ID: 243908
Assigned Server Name: null 20


### 3. Modification by Excel (Async injection with ID HUNTER)  (FAILED)

In [105]:
# Builds the database from scratch and physically guarantees the capture of the new ID.

import os
import time

print("--- PHASE 2 - BLOCK 3: SECURE ASYNC UPLOAD ---")

try:
    # 1. Get the highest ID currently in the project
    scenarios_before = api_instance.get_scenarios(the_project.id)
    max_id_before = max([s.id for s in scenarios_before]) if scenarios_before else 0
    print(f"Current highest Scenario ID before upload: {max_id_before}")
    
    if os.path.exists(modified_excel_path):
        print(f"\n2. Uploading file to create a BRAND NEW database...")
        import_response = api_instance.import_excel(
            new_scenario_name=f"AI_Scenario_Dec_{selected_decision_index}",
            project_id=the_project.id,
            file=modified_excel_path
        )
        
        job_id = getattr(import_response, 'job_id', None)
        
        if job_id:
            print(f"Server accepted the job. Job ID: {job_id}")
            print("3. Waiting for server to finish processing (Polling)...")
            
            finished = False
            while not finished:
                status_response = api_instance.get_import_status(job_id)
                current_status = getattr(status_response, 'status', 'UNKNOWN')
                
                if current_status == 'DONE':
                    finished = True
                elif current_status in ['FAILED', 'ERROR']:
                    print("[FATAL ERROR] AnyLogistix rejected the Excel file entirely!")
                    finished = True
                else:
                    time.sleep(3)
                    
            if current_status == 'DONE':
                print("\nJob marked as DONE. Hunting for the new Scenario ID...")
                
                # 4. The Hunter Loop: Wait until a new ID physically appears
                max_id_after = max_id_before
                attempts = 0
                
                while max_id_after <= max_id_before and attempts < 10:
                    time.sleep(2)
                    scenarios_after = api_instance.get_scenarios(the_project.id)
                    max_id_after = max([s.id for s in scenarios_after]) if scenarios_after else 0
                    attempts += 1
                    
                if max_id_after > max_id_before:
                    the_ai_scenario = [s for s in scenarios_after if s.id == max_id_after][0]
                    print(f"\n[SYSTEM TOTAL SUCCESS] New Scenario explicitly created and captured!")
                    print(f"Old ID: {max_id_before}  --->  NEW AI SCENARIO ID: {the_ai_scenario.id}")
                else:
                    print("\n[GHOST BUG] Server said DONE but no new scenario appeared.")
        else:
            print("\nError: Could not extract job_id from the server response.")
    else:
        print(f"Error: Could not find the file {modified_excel_path}")
        
except Exception as e:
    print("Exception during Excel upload: %s\n" % e)

--- PHASE 2 - BLOCK 3: SECURE ASYNC UPLOAD ---
Current highest Scenario ID before upload: 249011

2. Uploading file to create a BRAND NEW database...
Server accepted the job. Job ID: 99
3. Waiting for server to finish processing (Polling)...

Job marked as DONE. Hunting for the new Scenario ID...

[GHOST BUG] Server said DONE but no new scenario appeared.


### 3. Modification by Excel (FAILED)

In [131]:
# 4.a) Modification by Excel
# ROBUST SYNC INJECTION (CLONE + OVERWRITE)
# To bypass ALX server bugs when creating scenarios from scratch, the AI synchronously 
# clones the base scenario, and then synchronously overwrites it with the modified Excel.

import os
import openapi_client

print(f"--- PHASE 2 - BLOCK 3: CLONE AND OVERWRITE (DECISION {selected_decision_index}) ---")

try:
    if os.path.exists(modified_excel_path):
        print(f"1. Cloning base scenario '{the_scenario.name}' synchronously...")
        new_clone_name = f"{the_scenario.name} - AI Dec {selected_decision_index}"
        
        copy_parameters = openapi_client.ApiConvertAndCopyParameters(new_name=new_clone_name)
        
        cloned_scenario = api_instance.copy_scenario_synchronously(
            new_type=the_scenario.type, 
            scenario_id=the_scenario.id,
            body=copy_parameters
        )
        
        if cloned_scenario and cloned_scenario.id:
            print(f"   [SUCCESS] Scenario cloned! New ID: {cloned_scenario.id}")
            
            print(f"\n2. Injecting the modified Excel data into the clone synchronously...")
            # We use the existing import function which is bulletproof
            updated_scenario = api_instance.import_scenario_existing(
                project_id=the_project.id,
                scenario_id=cloned_scenario.id,
                file=modified_excel_path
            )
            
            print(f"\n[SYSTEM] TOTAL SUCCESS! AI data injected perfectly.")
            print(f"Final AI Scenario ID: {updated_scenario.id}")
            
            # Save it globally for the final simulation block
            the_ai_scenario = updated_scenario
            
        else:
            print("Error: Failed to clone the scenario. Server returned empty data.")
    else:
        print(f"Error: Could not find the file {modified_excel_path}")
        
except Exception as e:
    print("\n[SERVER ERROR] Exception during clone or upload:")
    print(e)

--- PHASE 2 - BLOCK 3: CLONE AND OVERWRITE (DECISION 1) ---
1. Cloning base scenario 'P4.2.6 SIM Distribution Network-4 Walls_New Inventory+New Transportation' synchronously...
   [SUCCESS] Scenario cloned! New ID: 257931

2. Injecting the modified Excel data into the clone synchronously...

[SYSTEM] TOTAL SUCCESS! AI data injected perfectly.
Final AI Scenario ID: 257931


### 4. Run simulation on AI scenario

In [132]:
# PHASE 2 - BLOCK 4: RUN SIMULATION ON AI SCENARIO
# The AI automatically configures a SIMULATION experiment for its newly created scenario, 
# and runs it synchronously to generate the results.

from pprint import pprint

print(f"--- PHASE 2 - BLOCK 4: RUNNING SIMULATION FOR AI SCENARIO ({the_ai_scenario.id}) ---")

try:
    print("1. Fetching SIMULATION configuration for the new AI scenario...")
    # Retrieve experiments explicitly for the AI scenario
    ai_run_configurations = api_instance.get_experiments(the_project.id, the_ai_scenario.id)
    
    # Select the 'SIMULATION' type
    ai_simulation_RC = next((r for r in ai_run_configurations if r.type == 'SIMULATION'), None)
    
    if ai_simulation_RC:
        print(f"Found SIMULATION experiment (ID: {ai_simulation_RC.id}). Launching synchronously...")
        
        # 2. Run the simulation
        sim_result = api_instance.run_experiment_synchronously(ai_simulation_RC.id)
        
        # Safely extract the result ID
        the_ai_sim_result_id = getattr(sim_result, 'experiment_result_id', None)
        
        if the_ai_sim_result_id:
            print(f"\n[SYSTEM] Simulation finished! Captured Result ID: {the_ai_sim_result_id}")
            
            # 3. Fetch the full list of dashboard pages (KPIs) and store them for the next block
            print("\n3. Fetching dashboard pages layout...")
            ai_dashboard_pages = api_instance.get_experiment_dashboard_pages(the_ai_sim_result_id)
            print(f"Found {len(ai_dashboard_pages)} dashboard pages ready for export.")
            
        else:
            print("Failed to capture 'experiment_result_id' from the synchronous run.")
            print("Server response was:")
            pprint(sim_result)
    else:
        print("No SIMULATION experiment found for the AI scenario.")
        
except Exception as e:
    print("Exception during Simulation Execution: %s\n" % e)

--- PHASE 2 - BLOCK 4: RUNNING SIMULATION FOR AI SCENARIO (257931) ---
1. Fetching SIMULATION configuration for the new AI scenario...
Found SIMULATION experiment (ID: 258618). Launching synchronously...

[SYSTEM] Simulation finished! Captured Result ID: 261435

3. Fetching dashboard pages layout...
Found 3 dashboard pages ready for export.


### 5. Export the new dashboard pages

In [133]:
# The AI loops through every available dashboard page (Finance, Inventory, etc.)
# and saves each one as a separate Excel file to preserve visual charts.

import shutil
import os
import re # Used for cleaning text strings

print(f"--- PHASE 2 - BLOCK 5: EXPORTING AI SIMULATION RESULTS ---")

try:
    if 'ai_dashboard_pages' in locals() and len(ai_dashboard_pages) > 0:
        print(f"Exporting ALL dashboard pages from AI result ID {the_ai_sim_result_id}...\n")
        
        # Iterate over every page available in the dashboard
        for page in ai_dashboard_pages:
            
            # Clean the page name to make it a safe Windows filename (replace spaces with underscores)
            safe_page_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', page.name)
            
            # Create a unique filename for each page, identifying it's from the AI decision
            output_filename = f"AI_Decision0_Results_{safe_page_name}_{the_ai_sim_result_id}.xlsx"
            output_path = os.path.join(os.getcwd(), output_filename)
            
            print(f"Downloading '{page.name}' (ID: {page.id}) -> {output_filename}")
            
            # Export this specific page
            excel_export = api_instance.export_dashboard_page(the_ai_sim_result_id, page.id)
            
            # Save the file securely
            if isinstance(excel_export, str) and os.path.exists(excel_export):
                shutil.move(excel_export, output_path)
            elif isinstance(excel_export, bytes):
                with open(output_path, "wb") as file:
                    file.write(excel_export)
            else:
                print(f"Unexpected data format received for page {page.name}")
                
        print("\n[TOTAL SUCCESS] All AI dashboard pages exported successfully.")
    else:
        print("No dashboard pages available to export. Did the simulation finish correctly?")
        
except Exception as e:
    print("Exception when exporting AI dashboard pages: %s\n" % e)

--- PHASE 2 - BLOCK 5: EXPORTING AI SIMULATION RESULTS ---
Exporting ALL dashboard pages from AI result ID 261435...


[TOTAL SUCCESS] All AI dashboard pages exported successfully.


### DIAGNOSTIC BLOCK

In [81]:
# DIAGNOSTIC BLOCK 2: INSPECT PATHS AND INVENTORY TABLES
import pandas as pd

base_excel_path = r"C:\Users\Luis\Downloads\TFG\API\Global Network Examination.xlsx" 

try:
    all_sheets = pd.read_excel(base_excel_path, sheet_name=None)
    
    print("--- 'Paths' TABLE ---")
    df_paths = all_sheets.get('Paths')
    if df_paths is not None and not df_paths.empty:
        print(df_paths.columns.tolist())
        print(df_paths.head(1).to_dict(orient='records')[0])
    else:
        print("Paths table not found.")
        
    print("\n--- 'Inventory' TABLE ---")
    df_inventory = all_sheets.get('Inventory')
    if df_inventory is not None and not df_inventory.empty:
        print(df_inventory.columns.tolist())
        print(df_inventory.head(1).to_dict(orient='records')[0])
    else:
        print("Inventory table not found.")
        
except Exception as e:
    print(f"Error: {e}")

--- 'Paths' TABLE ---
['ID', 'From', 'To', 'Cost Calculation', 'Col 1', 'Col 2', 'Col 3', 'Col 4', 'Col 5', 'Col 6', 'Currency', 'Distance', 'Distance Unit', 'Transportation Time', 'Col 7', 'Col 8', 'Time Unit', 'Straight', 'Vehicle Type', 'Time Period', 'Name', 'Inclusion Type']
{'ID': 'Path 1', 'From': 'Component supplier location', 'To': 'Factories', 'Cost Calculation': 'TransportationCostCalculatorVolumeDistance', 'Col 1': 'CO2 per unit', 'Col 2': 0, 'Col 3': 'Cost per unit', 'Col 4': 0.02, 'Col 5': 'Amount unit', 'Col 6': 'pcs', 'Currency': 'USD', 'Distance': 0, 'Distance Unit': 'km', 'Transportation Time': 'Value', 'Col 7': 'Value', 'Col 8': 0, 'Time Unit': 'day', 'Straight': False, 'Vehicle Type': 'Truck 1', 'Time Period': '(All periods)', 'Name': nan, 'Inclusion Type': 'Include'}

--- 'Inventory' TABLE ---
['ID', 'Facility', 'Product', 'Policy Type', 'Col 1', 'Col 2', 'Col 3', 'Col 4', 'Col 5', 'Col 6', 'Initial Stock, units', 'Periodic Check', 'Period', 'First Periodic Check',

### ==========================================================
## DEVELOPMENT TESTS - IGNORE THEM

In [ ]:
#Avoid self-signed warnings 
import os
import urllib3

For self-signed certificates it is necesary to donwload them and add to the certification path

In [ ]:
import ssl
import socket

def download_self_signed_cert_no_validation(hostname, port, cert_file):
    context = ssl._create_unverified_context()
    conn = context.wrap_socket(socket.socket(socket.AF_INET), server_hostname=hostname)
    conn.connect((hostname, port))
    der_cert = conn.getpeercert(binary_form=True)
    pem_cert = ssl.DER_cert_to_PEM_cert(der_cert)
    
    with open(cert_file, 'w') as cert_file:
        cert_file.write(pem_cert)
    
    print(f"Certificate saved to {cert_file.name}")

# Example usage
download_self_signed_cert_no_validation(SERVER_IP, 443, MY_CERT_FILENAME)

In [ ]:
# !/usr/bin/openssl s_client -showcerts -connect {SERVER_IP}:443 </dev/null 2>/dev/null | sed -n -e '/BEGIN\ CERTIFICATE/,/ENDCERTIFICATE/ p' > {MY_CERT_FILE}

In [ ]:


MY_CERT_FILE=os.path.abspath(MY_CERT_FILENAME)
if os.path.isfile(MY_CERT_FILE):
    print(f"Cert file OK: {MY_CERT_FILE} ")
else:
    print(f"Cert file ERROR: {MY_CERT_FILE} ")
MY_CERT_FILE

In [ ]:
os.environ['REQUESTS_CA_BUNDLE'] = MY_CERT_FILE

# Create a PoolManager with a custom CA bundle
http = urllib3.PoolManager(
    cert_reqs='CERT_NONE',      # Enforce cert verification
    ca_certs=MY_CERT_FILE           # Path to your self-signed cert
)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
http.request("GET", TEST_URL)

In [ ]:
from urllib3 import HTTPSConnectionPool

pool = HTTPSConnectionPool(
    SERVER_IP,
    port=443,
    cert_reqs='CERT_REQUIRED',
    ca_certs=MY_CERT_FILE
)

In [ ]:
import requests 

# Any request you make now will use your certificate
response = requests.get(TEST_URL, verify=False)
print(response.text)